## Converting to seconds

This notebook documents the process of converting AI-generated chart timestamps from beats to seconds.

### Problem

The DDC model outputs step timestamps in **beats**, but we need them in **seconds** for consistent analysis with human charts.

### Conversion

The DDC server generates charts at a fixed 125 BPM. To convert beats to seconds:

```
time_seconds = beat * 60 / 125
```

### Files
- **Input/Output**: `data/artificial/{fraxtil,itg}/*_ai.json` (modified in-place)
- Human charts already have `time` in seconds from the source data


In [ ]:
import json
from pathlib import Path

PROJECT_ROOT = Path('..').resolve()
AI_DIR = PROJECT_ROOT / 'data' / 'artificial'
HUMAN_DIR = PROJECT_ROOT / 'data' / 'real'
AI_BPM = 125.0

In [ ]:
def check_ai_charts_have_time():
    ai_files = list(AI_DIR.glob('**/*_ai.json'))
    if not ai_files:
        return False, 0, 0
    
    has_time = 0
    needs_conversion = 0
    
    for path in ai_files:
        with open(path, 'r') as f:
            data = json.load(f)
        for chart in data.get('charts', {}).values():
            steps = chart.get('steps', [])
            if steps:
                if 'time' in steps[0]:
                    has_time += 1
                elif 'beat' in steps[0]:
                    needs_conversion += 1
                break
    
    return has_time > 0 and needs_conversion == 0, has_time, needs_conversion

already_converted, has_time, needs_conv = check_ai_charts_have_time()
print(f"AI charts with 'time': {has_time}")
print(f"AI charts needing conversion: {needs_conv}")
print(f"Already converted: {already_converted}")

AI charts with 'time': 222
AI charts needing conversion: 0
Already converted: True


In [ ]:
def convert_ai_chart(path: Path) -> bool:
    with open(path, 'r') as f:
        data = json.load(f)
    
    modified = False
    for difficulty, chart in data.get('charts', {}).items():
        for step in chart.get('steps', []):
            if 'beat' in step and 'time' not in step:
                step['time'] = step.pop('beat') * 60 / AI_BPM
                modified = True
    
    if modified:
        with open(path, 'w') as f:
            json.dump(data, f, indent=2)
    
    return modified

In [ ]:
if already_converted:
    print(f"Skipping: All AI charts already have 'time' field")
else:
    ai_files = list(AI_DIR.glob('**/*_ai.json'))
    print(f"Processing {len(ai_files)} AI charts...")
    
    converted = 0
    for path in ai_files:
        if convert_ai_chart(path):
            converted += 1
    
    print(f"Converted {converted} AI charts to use seconds")

Skipping: All AI charts already have 'time' field


In [ ]:
def check_human_charts():
    human_files = list(HUMAN_DIR.glob('**/*_human.json'))
    needs_regen = []
    
    for path in human_files:
        with open(path, 'r') as f:
            data = json.load(f)
        for chart in data.get('charts', {}).values():
            steps = chart.get('steps', [])
            if steps and 'time' not in steps[0]:
                needs_regen.append(path)
                break
    
    return needs_regen

needs_regen = check_human_charts()
if needs_regen:
    print(f"{len(needs_regen)} human charts need regeneration.")
    print("Run notebook 01-normalize-human-charts.ipynb")
else:
    human_count = len(list(HUMAN_DIR.glob('**/*_human.json')))
    print(f"All {human_count} human charts have 'time' field!")

All 222 human charts have 'time' field!


In [ ]:
print("Sample AI chart step (after conversion):")
sample = next(AI_DIR.glob('**/*_ai.json'), None)
if sample:
    with open(sample, 'r') as f:
        data = json.load(f)
    first_chart = next(iter(data.get('charts', {}).values()), {})
    print(f"  File: {sample.name}")
    print(f"  First 3 steps: {first_chart.get('steps', [])[:3]}")

print("\nSample human chart step:")
sample = next(HUMAN_DIR.glob('**/*_human.json'), None)
if sample:
    with open(sample, 'r') as f:
        data = json.load(f)
    first_chart = next(iter(data.get('charts', {}).values()), {})
    print(f"  File: {sample.name}")
    print(f"  First 3 steps: {first_chart.get('steps', [])[:3]}")

Sample AI chart step (after conversion):
  File: Zodiac_ai.json
  First 3 steps: [{'step': '1000', 'time': 0.01}, {'step': '1000', 'time': 1.710000000000004}, {'step': '0010', 'time': 2.27}]

Sample human chart step:
  File: Why Me_human.json
  First 3 steps: [{'time': 8.796401871382127, 'step': '0001'}, {'time': 9.272176988343508, 'step': '0001'}, {'time': 9.747952105304892, 'step': '0001'}]
